# 07 · Colab DL 검증

ML과 같은 검증 7일·테스트 7일 `t`에서 `t+1`~`t+7`을 예측합니다. N-HiTS, TCN, TFT의 전체 학습 셀을 순서대로 배치했습니다. 실행 시간이 길 수 있으므로 필요한 모델 셀을 선택해 실행하세요.

`00` clone 노트북으로 저장소를 My Drive에 준비하고 GPU 런타임을 선택합니다. 원본 `data/raw/train.csv`와 ML 예측 CSV가 Drive에 있어야 합니다.


## 환경 준비


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/retail-demand-forecasting
%pip install -q -r requirements-neural.txt



## Library


In [ ]:
import logging
import sys
import warnings
from pathlib import Path
import pandas as pd

import seaborn as sns
import koreanize_matplotlib

REPO = Path('/content/drive/MyDrive/retail-demand-forecasting')
assert (REPO / 'src/retail_forecast/__init__.py').is_file(), f'저장소 경로를 확인하세요: {REPO}'
sys.path.insert(0, str(REPO / 'src'))

warnings.filterwarnings(
    'ignore',
    message=r'`isinstance\(treespec, LeafSpec\)` is deprecated.*',
    category=FutureWarning,
    module=r'pytorch_lightning\.utilities\._pytree',
)

from retail_forecast.comparison import compare_on_common_rows
from retail_forecast.neural_data import load_neural_panel, load_validation_template
from retail_forecast.neural_runner import inspect_series_schedule, run_neural_test, run_neural_validation

for logger_name in (
    'pytorch_lightning',
    'pytorch_lightning.accelerators.cuda',
    'pytorch_lightning.utilities.rank_zero',
    'lightning_fabric',
    'lightning_fabric.utilities.seed',
):
    logging.getLogger(logger_name).setLevel(logging.WARNING)


## 설정

학습에는 첫 검증 기준일 전의 판매량만 사용합니다. 각 기준일 `t`의 예측에는 `t`까지의 실제 판매 이력과 미리 아는 달력 변수만 사용합니다.


In [ ]:
sns.set_theme(style="whitegrid", font="NanumGothic")

ML_RUN_NAME = "full_history_time_split_fixed_i600"
ML_MODEL = "lightgbm"
VALID_TEMPLATE_PATH = REPO / "outputs" / ML_RUN_NAME / f"validation_predictions_{ML_MODEL}.csv"
TEST_TEMPLATE_PATH = REPO / "outputs" / ML_RUN_NAME / f"test_predictions_{ML_MODEL}.csv"
MODEL_STEPS = {"NHITS": 5000, "TCN": 5000, "TFT": 5000}
VAL_CHECK_STEPS = 250
DL_RUN_NAME = "neural_time_split_fixed"
INPUT_SIZE = 56
BATCH_SIZE = 32
WINDOWS_BATCH_SIZE = 256

assert VALID_TEMPLATE_PATH.is_file() and TEST_TEMPLATE_PATH.is_file()


## 학습 배치 확인

ML 검증 파일에서 시계열 수만 확인합니다. 모델이나 전체 일별 입력을 만들지 않고, 시계열 로더와 같은 방식으로 배치 순회를 재현합니다. 한 epoch의 step 수와 각 시계열의 선택 횟수를 확인합니다. 실제 모델 학습을 다시 실행하면 완료 로그에 Lightning의 실제 step 수와 epoch당 배치 수도 표시됩니다. `window_draws`는 날짜 창의 중복을 포함한 추출 횟수이며, 고유한 날짜 창 수가 아닙니다.


In [ ]:
valid_template = load_validation_template(VALID_TEMPLATE_PATH, expected_days=7)
test_template = load_validation_template(TEST_TEMPLATE_PATH, expected_days=7)
template = pd.concat([valid_template, test_template], ignore_index=True)
assert set(map(tuple, valid_template[["store_nbr", "family"]].drop_duplicates().to_numpy())) == set(
    map(tuple, test_template[["store_nbr", "family"]].drop_duplicates().to_numpy())
)
VALID_TRAIN_END = valid_template["date"].min() - pd.Timedelta(days=1)
series_count = len(template[["store_nbr", "family"]].drop_duplicates())
inspect_series_schedule(series_count, MODEL_STEPS["NHITS"], BATCH_SIZE, WINDOWS_BATCH_SIZE)


## ML 검증 기준일과 일별 입력

ML 결과에서 검증과 테스트 각각 7개의 연속 기준일과 1~7일 목표를 검사한다. 원본 판매 CSV에서 필요한 열만 읽는다. 결측 날짜는 0 판매로 간주하지 않고 `available_mask=0`으로 표시한다.

아래 셀은 전체 일별 입력을 준비하고 Parquet으로 저장합니다.


In [ ]:
full_panel, full_series = load_neural_panel(REPO / "data/raw", template)
processed = REPO / "data/processed"
processed.mkdir(parents=True, exist_ok=True)
full_panel.to_parquet(processed / "neural_panel_full.parquet", index=False)
full_series.to_parquet(processed / "neural_series_full.parquet", index=False)
print(f"검증·테스트 각 7개 기준일, {len(full_series)}개 시계열")


## 전체 모델 학습 · N-HiTS → TCN

학습 진행률이 표시됩니다. 이번 분할에서는 별도의 내부 검증을 쓰지 않고 외부 검증 7일의 지표를 확인합니다. 학습 손실 CSV와 검증·테스트 예측은 `outputs/neural_time_split_fixed`에, 모델은 대응하는 `models` 폴더에 저장됩니다. 학습 MSE는 정규화된 값이므로 평가 지표와 절댓값을 직접 비교하지 않습니다.


In [ ]:
nhits_valid = run_neural_validation(
    "NHITS", full_panel, full_series, valid_template,
    REPO / "outputs" / DL_RUN_NAME / "nhits", REPO / "models" / DL_RUN_NAME / "nhits_valid",
    input_size=INPUT_SIZE, max_steps=MODEL_STEPS["NHITS"],
    batch_size=BATCH_SIZE, windows_batch_size=WINDOWS_BATCH_SIZE,
    val_check_steps=VAL_CHECK_STEPS, train_end=VALID_TRAIN_END,
)
nhits_valid


In [ ]:
tcn_valid = run_neural_validation(
    "TCN", full_panel, full_series, valid_template,
    REPO / "outputs" / DL_RUN_NAME / "tcn", REPO / "models" / DL_RUN_NAME / "tcn_valid",
    input_size=INPUT_SIZE, max_steps=MODEL_STEPS["TCN"],
    batch_size=BATCH_SIZE, windows_batch_size=WINDOWS_BATCH_SIZE,
    val_check_steps=VAL_CHECK_STEPS, train_end=VALID_TRAIN_END,
)
tcn_valid


In [ ]:
common_paths = {
    "LightGBM": VALID_TEMPLATE_PATH,
    "CatBoost": REPO / "outputs" / ML_RUN_NAME / "validation_predictions_catboost.csv",
    "NHITS": REPO / "outputs" / DL_RUN_NAME / "nhits/validation_predictions_nhits.parquet",
    "TCN": REPO / "outputs" / DL_RUN_NAME / "tcn/validation_predictions_tcn.parquet",
}
validation_table = compare_on_common_rows(common_paths)
validation_table.to_csv(REPO / "outputs" / DL_RUN_NAME / "comparison_ml_dl_validation.csv", index=False)
validation_table


## TFT

N-HiTS와 TCN의 전체 결과를 확인한 후 이 셀을 실행합니다.


In [ ]:
tft_valid = run_neural_validation(
    "TFT", full_panel, full_series, valid_template,
    REPO / "outputs" / DL_RUN_NAME / "tft", REPO / "models" / DL_RUN_NAME / "tft_valid",
    input_size=INPUT_SIZE, max_steps=MODEL_STEPS["TFT"],
    batch_size=BATCH_SIZE, windows_batch_size=WINDOWS_BATCH_SIZE,
    val_check_steps=VAL_CHECK_STEPS, train_end=VALID_TRAIN_END,
)
tft_valid


In [ ]:
common_paths["TFT"] = REPO / "outputs" / DL_RUN_NAME / "tft/validation_predictions_tft.parquet"
validation_table = compare_on_common_rows(common_paths)
validation_table.to_csv(REPO / "outputs" / DL_RUN_NAME / "comparison_ml_dl_validation.csv", index=False)
validation_table


## 최종 테스트
위 검증표로 설정을 선택합니다. `MODEL_STEPS`를 바꿨다면 해당 모델의 검증 학습 셀을 다시 실행해 저장 모델을 갱신하세요. 아래 셀은 저장한 동일 모델로 예측만 수행합니다.


In [ ]:
nhits_test = run_neural_test(
    "NHITS", full_panel, full_series, test_template,
    REPO / "outputs" / DL_RUN_NAME / "nhits", REPO / "models" / DL_RUN_NAME / "nhits_valid",
    input_size=INPUT_SIZE,
)
nhits_test


In [ ]:
tcn_test = run_neural_test(
    "TCN", full_panel, full_series, test_template,
    REPO / "outputs" / DL_RUN_NAME / "tcn", REPO / "models" / DL_RUN_NAME / "tcn_valid",
    input_size=INPUT_SIZE,
)
tcn_test


In [ ]:
tft_test = run_neural_test(
    "TFT", full_panel, full_series, test_template,
    REPO / "outputs" / DL_RUN_NAME / "tft", REPO / "models" / DL_RUN_NAME / "tft_valid",
    input_size=INPUT_SIZE,
)
tft_test


In [ ]:
test_paths = {
    "LightGBM": TEST_TEMPLATE_PATH,
    "CatBoost": REPO / "outputs" / ML_RUN_NAME / "test_predictions_catboost.csv",
    "NHITS": REPO / "outputs" / DL_RUN_NAME / "nhits/test_predictions_nhits.parquet",
    "TCN": REPO / "outputs" / DL_RUN_NAME / "tcn/test_predictions_tcn.parquet",
    "TFT": REPO / "outputs" / DL_RUN_NAME / "tft/test_predictions_tft.parquet",
}
test_table = compare_on_common_rows(test_paths)
test_table.to_csv(REPO / "outputs" / DL_RUN_NAME / "comparison_ml_dl_test.csv", index=False)
test_table
